# 06 — Extracción de indicadores V-Dem relevantes

**Objetivo del notebook:** construir `df_VDEM_base`, un panel país-año (1960–2024) con los 79 índices intermedios de V-Dem seleccionados en `metadatavdem_79_indices_intermedios.xlsx`, restringido a los 193 países miembros de la ONU (`df_regiones_miembros_onu.xlsx`).


**Decisiones metodológicas ya acordadas para este cuaderno:**
- Rango temporal de descarga completo: 1960–2024 
- Universo de países: los 193 miembros de la ONU (`cow_code_countryVdem`).



## Bloque 1 — Setup y rutas

**Objetivo:** cargar librerías y definir las rutas a los tres insumos.


In [1]:
import pandas as pd
import numpy as np
import time
import os

# Rutas — ajusta VDEM_PATH a tu ubicación local del CSV
ONU_PATH = "df_regiones_miembros_onu.xlsx"
METADATA_VDEM_79_PATH = "metadatavdem_79_indices_intermedios.xlsx"
VDEM_PATH = "V-Dem-CY-Full+Others-v16.csv"  # <-- ajustar si tu archivo está en otra carpeta

ANIO_MIN = 1960
ANIO_MAX = 2024


## Bloque 2 — Cargar `df_regiones_miembros_onu.xlsx`

**Objetivo:** cargar la tabla de los 193 países miembros de la ONU y quedarnos únicamente con las columnas necesarias para este cuaderno.

**Justificación metodológica:** de las 13 columnas originales, solo necesitamos las identificatorias/regionales acordadas más la clave de unión (`cow_code_countryVdem`). Restringir columnas desde el inicio evita arrastrar información no solicitada.

**Resultado esperado:** `df_onu_base` con 193 filas y 7 columnas: `Country or Area`, `M49_region`, `Region Name`, `M49_subregion`, `Sub-region Name`, `ISO-alpha2 Code`, `cow_code_countryVdem`.


In [2]:
df_onu_raw = pd.read_excel(ONU_PATH)

columnas_onu_necesarias = [
    "Country or Area",
    "M49_region",
    "Region Name",
    "M49_subregion",
    "Sub-region Name",
    "ISO-alpha2 Code",
    "cow_code_countryVdem",
]

df_onu_base = df_onu_raw[columnas_onu_necesarias].copy()

print(f"Shape df_onu_base: {df_onu_base.shape}")
print(f"Nulos en cow_code_countryVdem: {df_onu_base['cow_code_countryVdem'].isna().sum()}")
df_onu_base.head()


Shape df_onu_base: (193, 7)
Nulos en cow_code_countryVdem: 0


,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code,cow_code_countryVdem
0,United States of America,19,Americas,21,Northern America,US,2
1,Australia,9,Oceania,53,Australia and New Zealand,AU,900
2,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ,522
3,Ghana,2,Africa,202,Sub-Saharan Africa,GH,452
4,Kiribati,9,Oceania,57,Micronesia,KI,946


**Qué comprobar:** que `df_onu_base.shape` sea `(193, 7)` y que no haya nulos en `cow_code_countryVdem` (ya lo confirmamos en la fase de discusión, pero se revalida acá por transparencia).




## Bloque 3 — Cargar los 79 códigos V-Dem a extraer

**Objetivo:** cargar `metadatavdem_79_indices_intermedios.xlsx` y extraer la lista de códigos de la columna `columna`.

**Justificación metodológica:** esta lista es la que usaremos como filtro de columnas al leer el CSV de V-Dem (79 índices intermedios `v2x*`, ya validados sin duplicados en la fase de discusión previa).

**Resultado esperado:** una lista `codigos_79` con 79 elementos, sin duplicados.


In [3]:
df_metadata_79 = pd.read_excel(METADATA_VDEM_79_PATH)

codigos_79 = df_metadata_79["columna"].tolist()

assert len(codigos_79) == len(set(codigos_79)), "Hay codigos duplicados en metadatavdem_79_indices_intermedios.xlsx"

print(f"Total de codigos a extraer: {len(codigos_79)}")
print(codigos_79[:10], "...")


Total de codigos a extraer: 79
['v2x_suffr', 'v2x_jucon', 'v2xlg_legcon', 'v2x_cspart', 'v2xdd_dd', 'v2xel_locelec', 'v2xel_regelec', 'v2xdl_delib', 'v2xeg_eqaccess', 'v2xeg_eqdr'] ...


**Qué comprobar:** que el total sea 79 y que el `assert` no falle.



## Bloque 4 — Verificar cabecera de V-Dem y construir `usecols`

**Objetivo:** leer solo la cabecera del CSV de V-Dem (`nrows=0`, sin cargar datos), confirmar que los 79 códigos existen tal cual en el archivo, y construir la lista final de columnas a leer (`usecols`).

**Justificación metodológica:** el archivo pesa ~397MB con miles de columnas. Antes de intentar un `usecols` que podría fallar con un error poco claro si algún código no existe, verificamos explícitamente cuáles códigos están presentes y cuáles no. Esto es más seguro que asumir que los 79 nombres coinciden exactamente con los del CSV real.

**Resultado esperado:** confirmación de que los 79 códigos existen (o, si no, la lista exacta de los que faltan), y una lista `usecols_finales` con identificatorias + los 79 códigos.


In [4]:
t0 = time.time()
columnas_vdem_header = pd.read_csv(VDEM_PATH, nrows=0).columns.tolist()
t1 = time.time()

print(f"Total columnas en el CSV: {len(columnas_vdem_header)}")
print(f"Tiempo de lectura de cabecera: {t1 - t0:.2f} segundos")

columnas_identificatorias = ["country_name", "COWcode", "year", "project", "historical"]

faltantes_identificatorias = [c for c in columnas_identificatorias if c not in columnas_vdem_header]
faltantes_codigos = [c for c in codigos_79 if c not in columnas_vdem_header]

print(f"Identificatorias faltantes: {faltantes_identificatorias}")
print(f"Codigos de los 79 faltantes en el CSV: {faltantes_codigos}")

if faltantes_identificatorias or faltantes_codigos:
    print("\n⚠️ Hay columnas solicitadas que no existen en el CSV. Revisar antes de continuar.")
else:
    print("\n✅ Todas las columnas solicitadas existen en el CSV.")

usecols_finales = columnas_identificatorias + codigos_79


Total columnas en el CSV: 4618
Tiempo de lectura de cabecera: 0.41 segundos
Identificatorias faltantes: []
Codigos de los 79 faltantes en el CSV: []

✅ Todas las columnas solicitadas existen en el CSV.


**Qué comprobar:** que aparezca el mensaje "✅ Todas las columnas solicitadas existen en el CSV". Si aparece el mensaje de advertencia, **no continúes al bloque 5** — compárteme la lista de `faltantes_codigos` para revisar si son variantes de nombre (ej. mayúsculas, sufijo distinto) o si realmente hay que ajustar la selección de indicadores.

**Errores habituales:** si `country_name` o `COWcode` no aparecen con ese nombre exacto (podría ser `e_v2x_...` u otra convención en v16), lo vemos en `faltantes_identificatorias` y ajustamos.


## Bloque 5 — Lectura filtrada del CSV y recorte temporal

**Objetivo:** leer `V-Dem-CY-Full+Others-v16.csv` cargando únicamente las columnas de `usecols_finales`, y filtrar filas al rango de años 1960–2024.



**Resultado esperado:** `df_vdem_filtrado` con las columnas de `usecols_finales` y solo filas con `year` entre 1960 y 2024.


In [5]:
t0 = time.time()
df_vdem_raw = pd.read_csv(VDEM_PATH, usecols=usecols_finales, low_memory=False)
t1 = time.time()

print(f"Shape tras lectura (usecols aplicado): {df_vdem_raw.shape}")
print(f"Tiempo de carga: {t1 - t0:.1f} segundos")
print(f"Memoria usada: {df_vdem_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

df_vdem_filtrado = df_vdem_raw[
    (df_vdem_raw["year"] >= ANIO_MIN) & (df_vdem_raw["year"] <= ANIO_MAX)
].copy()

print(f"\nShape tras filtro de años ({ANIO_MIN}-{ANIO_MAX}): {df_vdem_filtrado.shape}")
print(f"Rango de anios resultante: {df_vdem_filtrado['year'].min()} - {df_vdem_filtrado['year'].max()}")


Shape tras lectura (usecols aplicado): (28092, 84)
Tiempo de carga: 11.2 segundos
Memoria usada: 19.1 MB

Shape tras filtro de años (1960-2024): (10908, 84)
Rango de anios resultante: 1960 - 2024


**Qué comprobar:** que el número de columnas de `df_vdem_raw` sea igual a `len(usecols_finales)` (84: 5 identificatorias + 79 códigos). Que el rango de años tras el filtro efectivamente esté dentro de 1960–2024.

**Errores habituales:** `MemoryError` — aunque poco probable con solo 84 columnas, si tu máquina tiene RAM muy limitada sem puede usar `chunksize` como alternativa. `DtypeWarning` por columnas mixtas es posible y no crítico en esta fase.


## Bloque 6 — Reporte de duplicados `(COWcode, year)`

**Objetivo:** medir empíricamente cuántas combinaciones `(COWcode, year)` aparecen más de una vez en `df_vdem_filtrado`, y describir el patrón (qué valores toman `project`/`historical` en esos casos).

**Justificación metodológica:** V-Dem "Full+Others" combina el proyecto contemporáneo con extensiones históricas, y algunos códigos COW se reutilizan para entidades políticas sucesivas. Antes de decidir cómo tratar estos duplicados (fuera del alcance de este cuaderno), necesitamos evidencia concreta de si el problema existe y de qué magnitud es. Por ahora **no se eliminan ni se resuelven** — se documentan.

**Resultado esperado:** `df_duplicados_reporte`, con una fila por cada combinación `(COWcode, year)` duplicada, y un conteo total.


In [6]:
conteo_por_pais_anio = (
    df_vdem_filtrado.groupby(["COWcode", "year"])
    .size()
    .reset_index(name="n_filas")
)

combinaciones_duplicadas = conteo_por_pais_anio[conteo_por_pais_anio["n_filas"] > 1]

print(f"Combinaciones (COWcode, year) duplicadas: {combinaciones_duplicadas.shape[0]}")
print(f"Total de filas afectadas: {combinaciones_duplicadas['n_filas'].sum()}")

if not combinaciones_duplicadas.empty:
    df_duplicados_reporte = df_vdem_filtrado.merge(
        combinaciones_duplicadas[["COWcode", "year"]],
        on=["COWcode", "year"],
        how="inner",
    ).sort_values(["COWcode", "year"])

    print("\nDistribucion de 'project' en filas duplicadas:")
    print(df_duplicados_reporte["project"].value_counts(dropna=False))
    print("\nDistribucion de 'historical' en filas duplicadas:")
    print(df_duplicados_reporte["historical"].value_counts(dropna=False))

    print("\nEjemplos (primeras 10 filas):")
    display(df_duplicados_reporte[["country_name", "COWcode", "year", "project", "historical"]].head(10))
else:
    df_duplicados_reporte = pd.DataFrame(columns=df_vdem_filtrado.columns)
    print("No se encontraron duplicados de (COWcode, year) en el rango 1960-2024.")


Combinaciones (COWcode, year) duplicadas: 0
Total de filas afectadas: 0
No se encontraron duplicados de (COWcode, year) en el rango 1960-2024.


**Qué comprobar:** si hay duplicados, revisa los ejemplos — típicamente vas a ver `project` con valores distintos (ej. contemporáneo vs. histórico) para el mismo país-año. Guarda mentalmente este patrón porque será la base de la decisión metodológica pendiente (resolver duplicados) cuando integres las fuentes.

**Errores habituales:** ninguno esperado; si `combinaciones_duplicadas` está vacío, es un resultado válido (no hay que "forzar" que aparezcan duplicados).


## Bloque 7 — Merge con `df_onu_base` (left join)

**Objetivo:** unir los países de df_onu_base con df_vdem_filtrado, usando cow_code_countryVdem == COWcode.

**Justificación metodológica:** Aquí buscamos extraer datos solo para los países que figuran en df_regiones_miembros_onu.xlsx — es una restricción de que no entre nada fuera de esa lista, no una garantía de que los 193 tengan que aparecer sí o sí. Si un país ONU no tiene ninguna fila en V-Dem, simplemente no debe figurar en df_VDEM_base;

**Resultado esperado:**  df_VDEM_base con únicamente los países ONU que sí tienen al menos una fila en V-Dem, para el rango 1960–2024.


In [7]:
df_VDEM_base = df_onu_base.merge(
    df_vdem_filtrado,
    left_on="cow_code_countryVdem",
    right_on="COWcode",
    how="inner",
)

print(f"Shape df_VDEM_base: {df_VDEM_base.shape}")
print(f"Paises unicos (por cow_code_countryVdem): {df_VDEM_base['cow_code_countryVdem'].nunique()} de 193")

Shape df_VDEM_base: (10398, 91)
Paises unicos (por cow_code_countryVdem): 172 de 193


## Bloque 8 — Renombrar y ordenar columnas finales

**Objetivo:** renombrar `country_name` (V-Dem) a `country_name_VDEM`, y ordenar las columnas del `df_VDEM_base` final: identificatorias primero, luego los 79 indicadores.

**Justificación metodológica:** `country_name_VDEM` evita ambigüedad con `Country or Area` (el nombre de país de tu tabla ONU) cuando más adelante integres otras fuentes. Mantener `COWcode` explícito permite auditar el merge en cualquier momento posterior.

**Resultado esperado:** `df_VDEM_base` con columnas en este orden: `Country or Area`, `ISO-alpha2 Code`, `M49_region`, `Region Name`, `M49_subregion`, `Sub-region Name`, `cow_code_countryVdem`, `COWcode`, `country_name_VDEM`, `year`, `project`, `historical`, seguido de los 79 códigos.


In [8]:
df_VDEM_base = df_VDEM_base.rename(columns={"country_name": "country_name_VDEM"})

columnas_orden_final = [
    "Country or Area",
    "ISO-alpha2 Code",
    "M49_region",
    "Region Name",
    "M49_subregion",
    "Sub-region Name",
    "cow_code_countryVdem",
    "COWcode",
    "country_name_VDEM",
    "year",
    "project",
    "historical",
] + codigos_79

df_VDEM_base = df_VDEM_base[columnas_orden_final]

print(f"Shape final: {df_VDEM_base.shape}")
df_VDEM_base.head()


Shape final: (10398, 91)


,Country or Area,ISO-alpha2 Code,M49_region,Region Name,M49_subregion,Sub-region Name,cow_code_countryVdem,COWcode,country_name_VDEM,year,...,v2xpas_economic_opposition,v2xed_ed_poed,v2xed_ed_cent,v2xed_ed_ctag,v2xed_ed_con,v2xed_ed_dmcon,v2xed_ed_ptcon,v2xed_ptcon,v2xedvd_me_cent,v2xedvd_me_ctag
0,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1960,...,NaN,0.258,0.003,0.219,0.811,0.831,0.734,0.646,0.181,0.132
1,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1961,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
2,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1962,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
3,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1963,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
4,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1964,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132


**Qué comprobar:** que el número de columnas sea `12 + 79 = 91`. Revisa visualmente algunas filas para confirmar que `Country or Area` y `country_name_VDEM` correspondan al mismo país (ej. Argentina ↔ Argentina), como chequeo de sanidad del merge.

**Errores habituales:** ninguno esperado; es una operación de renombrado y reindexado de columnas ya existentes.


## Bloque 9 — Países ONU sin ninguna fila en V-Dem

OBJETIVO: identificar los países de los 193 que quedaron fuera de df_VDEM_base tras el inner join.

**Resultado esperado:** `df_paises_sin_vdem`, listado de países ONU sin ninguna fila de V-Dem.


In [13]:
#Antes de descargar la base, vamos a eliminar columnas que son irrelevantes para los fines de este trabajo, o que son redundantes
df_VDEM_base = df_VDEM_base.drop(columns=["COWcode", "project", "historical"])

print(f"Shape tras eliminar columnas: {df_VDEM_base.shape}")
df_VDEM_base.head()

Shape tras eliminar columnas: (10398, 88)


,Country or Area,ISO-alpha2 Code,M49_region,Region Name,M49_subregion,Sub-region Name,cow_code_countryVdem,country_name_VDEM,year,v2x_suffr,...,v2xpas_economic_opposition,v2xed_ed_poed,v2xed_ed_cent,v2xed_ed_ctag,v2xed_ed_con,v2xed_ed_dmcon,v2xed_ed_ptcon,v2xed_ptcon,v2xedvd_me_cent,v2xedvd_me_ctag
0,United States of America,US,19,Americas,21,Northern America,2,United States of America,1960,0.95,...,NaN,0.258,0.003,0.219,0.811,0.831,0.734,0.646,0.181,0.132
1,United States of America,US,19,Americas,21,Northern America,2,United States of America,1961,0.95,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
2,United States of America,US,19,Americas,21,Northern America,2,United States of America,1962,0.95,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
3,United States of America,US,19,Americas,21,Northern America,2,United States of America,1963,0.95,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
4,United States of America,US,19,Americas,21,Northern America,2,United States of America,1964,0.95,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132


## Bloque 10 — Exportar a Excel

**Objetivo:** guardar `df_VDEM_base`, junto con el reporte de duplicados y el listado de países sin cobertura, en un único archivo Excel multi-hoja.

**Justificación metodológica:** mantener la documentación de las dos limitaciones detectadas (duplicados, cobertura) junto con los datos, siguiendo tu práctica de documentar decisiones metodológicas relevantes en los archivos de salida.

**Resultado esperado:** `df_VDEM_base.xlsx` en tu directorio de trabajo local, con 3 hojas: `df_VDEM_base`, `duplicados_COWcode_year`, `paises_sin_cobertura_VDEM`.


In [ ]:
df_VDEM_base = df_VDEM_base.drop(columns=["COWcode", "project", "historical"])

print(f"Shape tras eliminar columnas: {df_VDEM_base.shape}")
df_VDEM_base.head()
df_VDEM_base.head()

,Country or Area,ISO-alpha2 Code,M49_region,Region Name,M49_subregion,Sub-region Name,cow_code_countryVdem,COWcode,country_name_VDEM,year,...,v2xpas_economic_opposition,v2xed_ed_poed,v2xed_ed_cent,v2xed_ed_ctag,v2xed_ed_con,v2xed_ed_dmcon,v2xed_ed_ptcon,v2xed_ptcon,v2xedvd_me_cent,v2xedvd_me_ctag
0,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1960,...,NaN,0.258,0.003,0.219,0.811,0.831,0.734,0.646,0.181,0.132
1,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1961,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
2,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1962,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
3,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1963,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
4,United States of America,US,19,Americas,21,Northern America,2,2.0,United States of America,1964,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132


In [14]:
with pd.ExcelWriter("df_VDEM_base.xlsx", engine="openpyxl") as writer:
    df_VDEM_base.to_excel(writer, sheet_name="df_VDEM_base", index=False)
    df_duplicados_reporte[["country_name", "COWcode", "year", "project", "historical"]].to_excel(
        writer, sheet_name="duplicados_COWcode_year", index=False
    )
    df_paises_sin_vdem.to_excel(writer, sheet_name="paises_sin_cobertura_VDEM", index=False)

print("Archivo exportado: df_VDEM_base.xlsx")


Archivo exportado: df_VDEM_base.xlsx


**Qué comprobar:** abre el Excel y verifica las 3 hojas. En `df_VDEM_base`, confirma que el orden de columnas sea el esperado y que los valores de los indicadores (ej. `v2x_polyarchy_stock`) tengan sentido (rango de valores plausible, no todo NaN salvo en los países del bloque 9).

**Errores habituales:** si `openpyxl` no está instalado, `pip install openpyxl`. Si el archivo es muy grande y tarda, es esperable dado el volumen (193+ países × ~65 años × 91 columnas).
